# Solar filament instance segmentation: end-to-end pipeline

Test images -> Mask R-CNN detector (translate TTA, mask NMS, score threshold, pixel ownership)
-> seed-box crop U-Net refiner -> disjoint instance masks -> submission CSV, then every quantity of
the committee rubric on the 60-image validation split. All steps call `filseg/`; the same code runs
from the command line as `scripts/predict.py` and `scripts/rubric_metrics.py`.

The two checkpoints are in this repository (`weights/`, Git LFS; run `git lfs pull` if the first code cell
reports pointer files). The challenge data goes under `data/`
(`kaggle competitions download -c filament-segmentation-2026 -p data --unzip`), or set `FILSEG_DATA`.
Without it the notebook writes one synthetic full-disk image and skips the rubric section.
`FILSEG_LIMIT=N` keeps the first N images of each split.


In [ ]:
import hashlib, os, sys
from pathlib import Path

import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from filseg.infer import Config, encode, predict_all, submission_rows
from filseg.io import heldout_stems, load_records
from filseg.metrics import evaluate
from filseg.model import load_detector, load_refiner

DATA = Path(os.environ.get("FILSEG_DATA", ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026"))
LIMIT = int(os.environ["FILSEG_LIMIT"]) if os.environ.get("FILSEG_LIMIT") else None
TEST_IMAGES = DATA / "test" / "test_images"
TRAIN_IMAGES = DATA / "train" / "train_images"
ANN = DATA / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
DETECTOR, REFINER = ROOT / "weights" / "detector.pt", ROOT / "weights" / "refiner.pt"
for line in (ROOT / "weights" / "SHA256SUMS").read_text().splitlines():
    digest, name = line.split()
    with open(ROOT / "weights" / name, "rb") as f:
        if f.read(24).startswith(b"version https://git-lfs"):
            raise RuntimeError(f"weights/{name} is a Git LFS pointer: run `git lfs install && git lfs pull`")
        f.seek(0)
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise RuntimeError(f"weights/{name} does not match weights/SHA256SUMS")
print("weights verified against SHA256SUMS")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config()  # score 0.80, refine 0.40, min area 250 px, NMS IoU 0.1, TTA shift 16 px
cfg, device

In [ ]:
detector = load_detector(DETECTOR, device)
refiner = load_refiner(REFINER, device)

## 1. Test set -> submission CSV

In [ ]:
paths = sorted(TEST_IMAGES.glob("*.jpeg"))[:LIMIT]
if not paths:  # no challenge data: one synthetic full-disk image with dark curved filaments
    import numpy as np
    from PIL import Image

    yy, xx = np.mgrid[:2048, :2048]
    disk = np.hypot(xx - 1024, yy - 1024) < 900
    img = np.where(disk, 150.0, 5.0) + np.random.default_rng(0).normal(0, 3, disk.shape)
    for x0, y0, a, phase in [(600, 700, 60, 0.0), (1100, 1200, 40, 1.0), (1300, 600, 50, 2.0)]:
        for x in range(x0, x0 + 400):
            y = y0 + a * np.sin(x / 80 + phase)
            img[int(y) - 9 : int(y) + 9, x] = 70
    sample = ROOT / "outputs" / "sample_input"
    sample.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.clip(img, 0, 255).astype("uint8")).save(sample / "synthetic_0.jpeg", quality=95)
    paths = [sample / "synthetic_0.jpeg"]
    print("no challenge data found: running on one synthetic image")
rows, seconds = [], []
for stem, masks, shape, dt in predict_all(paths, detector, refiner, cfg, device):
    rows += submission_rows(stem, masks, shape)
    seconds.append(dt)
out = ROOT / "outputs" / "submission.csv"
out.parent.mkdir(exist_ok=True)
out.write_text("filament_id,segmentation_rle\n" + "\n".join(rows) + "\n")
print(f"{out}: {len(rows)} rows, {len(paths)} images, {sum(seconds):.0f} s ({sum(seconds)/len(paths):.2f} s/image on {device})")

## 2. Validation split -> rubric metrics (PQ over every annotator record, IoU/Dice, 1:n and n:1)

In [ ]:
if ANN.exists() and TRAIN_IMAGES.is_dir():
    val = heldout_stems(ANN)
    val_paths = [p for p in sorted(TRAIN_IMAGES.glob("*.jpeg")) if p.stem in val][:LIMIT]
    preds = {}
    for stem, masks, shape, dt in predict_all(val_paths, detector, refiner, cfg, device):
        preds[stem] = [{"size": list(shape), "counts": encode(m).encode()} for m in masks]
    summary = evaluate(load_records(ANN, {p.stem for p in val_paths}), preds).summary()
else:
    summary = "rubric section skipped: no challenge annotations under " + str(DATA)
summary


## 3. Final selection (CPRVOTE_G): five-network vote -> guarded CPR pass

The submitted file is the two scripts below run in sequence (`scripts/predict_vote.py`, then `scripts/refine_cpr.py`); on all 180 test images the output MD5 is `3c92671d9463ce2466ef418ae0609fbd` (about 2.7 h on one RTX PRO 6000). `FILSEG_LIMIT=N` keeps the first N images. Without challenge test images the section is skipped.

In [ ]:
import subprocess

FINAL = ROOT / "outputs" / "cprvote_G.csv"
if TEST_IMAGES.is_dir():
    lim = ["--limit", str(LIMIT)] if LIMIT else []
    vote_csv = ROOT / "outputs" / "vote.csv"
    subprocess.run([sys.executable, str(ROOT / "scripts" / "predict_vote.py"), "--images", str(TEST_IMAGES),
                    "--weights", str(ROOT / "weights"), "--out", str(vote_csv), *lim], check=True)
    subprocess.run([sys.executable, str(ROOT / "scripts" / "refine_cpr.py"), "--images", str(TEST_IMAGES),
                    "--csv", str(vote_csv), "--weights", str(ROOT / "weights" / "cpr.pt"), "--out", str(FINAL), *lim],
                   check=True, env={**os.environ, "OMP_NUM_THREADS": "1"})
    final = f"{FINAL}: md5 {hashlib.md5(FINAL.read_bytes()).hexdigest()} (submitted: 3c92671d9463ce2466ef418ae0609fbd)"
else:
    final = "final-selection section skipped: no challenge test images under " + str(DATA)
final